# Segmentação de pluma de fumaça com SAM — versão aprimorada

Continuação do notebook inicial da atividade: mesma stack (**Ultralytics**), *prompts* e avaliação
reescritos. A imagem do notebook inicial (`MyDrive/pluma.jpg`) não está mais disponível, então a
análise usa uma imagem pública equivalente, com *prompts* e máscara de referência já calibrados.

## O que estava quebrado no notebook inicial

| Célula | O que acontecia | Causa | O que foi feito aqui |
|---|---|---|---|
| 13 | `FileNotFoundError: 'sam_b.pt'` | `SAM3SemanticPredictor` espera pesos do **SAM 3**; o download deu 404 e `sam_b.pt` (SAM 1) não faz *prompt* de texto | `SAM2Predictor` com *prompts* **geométricos** (caixa e pontos), que é o que a tarefa pede |
| 14 | `(no detections)` → "Nenhuma fumaça detectada" | o detector `YOLOv8m-smoke-detection` não achou a pluma, então **nenhuma máscara foi gerada** | não dependo de detector automático: a caixa e os pontos são definidos na imagem |
| 12 | YOLO genérico acusou "4 cars, 1 truck" | `yolo26n` é COCO: não existe classe "fumaça" | removido — não contribui para a máscara |
| 7 | `shutil.copyfile("pluma.jpg", "pluma.png")` | só renomeia: o arquivo continua **JPEG** com extensão `.png` | leio o original direto, sem cópia |

> ⚠️ **A célula 4 do notebook inicial tem uma chave da API do CDS/Copernicus escrita no código.**
> Chave em repositório público é chave vazada: revogue essa e gere outra no painel do CDS, e passe a
> lê-la de variável de ambiente (`os.environ["CDSAPI_KEY"]`). Este notebook não contém a chave nem a
> parte de ERA5/netCDF — aquilo é outra frente do projeto; aqui é só a segmentação da pluma.

## A imagem

*Smoke plume from SoCal wildfire, outubro/2007* (Wikimedia Commons, domínio público). Foi escolhida
porque reúne **os três distratores citados no enunciado ao mesmo tempo**: céu azul, casas/telhados e
uma faixa de *haze* esbranquiçada no horizonte quase indistinguível da base da pluma — e ainda uma
placa de trânsito que **oclui** parte da pluma.

Para usar outra imagem: `CFG["IMAGEM"] = "upload"` (ou um caminho/URL). Nesse caso os *prompts*
calibrados não valem mais, e a Etapa 1 calcula um palpite automático e mostra a imagem com uma grade
de coordenadas para você ajustar `PROMPTS` na Etapa 2.

**Como rodar:** `Ambiente de execução → Executar tudo`. GPU ajuda mas não é obrigatória (usa
`sam2_t.pt` em CPU e `sam2_b.pt` em GPU).

In [ ]:
# ============================================================
# ETAPA 1 — Instalação, imagem e prompts
# ============================================================
!pip install -q ultralytics

import os, urllib.request
import numpy as np, cv2, torch, pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle

np.random.seed(42); torch.manual_seed(42)
plt.rcParams.update({"figure.dpi": 110, "figure.facecolor": "white",
                     "axes.titlesize": 10, "axes.titleweight": "bold", "font.size": 9})
os.makedirs("saida", exist_ok=True)

CFG = {"IMAGEM": "socal"}        # "socal" | "upload" | caminho ou URL de outra imagem
DISPOSITIVO = "cuda" if torch.cuda.is_available() else "cpu"
MODELO = "sam2_b.pt" if DISPOSITIVO == "cuda" else "sam2_t.pt"   # Ultralytics baixa sozinho

# Prompts em coordenadas FRACIONÁRIAS (x/largura, y/altura): continuam válidos em qualquer
# resolução, então a mesma lista serve para a imagem original e para a versão reescalada.
CALIBRADO = {
    "url": ("https://commons.wikimedia.org/wiki/Special:FilePath/"
            "Smoke%20plume%20from%20SoCal%20wildfire%20October%202007.jpg?width=1280"),
    "caixa": (0.289, 0.266, 0.999, 0.625),
    "positivos": [(0.336, 0.479), (0.484, 0.438), (0.625, 0.385), (0.742, 0.344), (0.320, 0.531)],
    "negativos": [(0.117, 0.156), (0.547, 0.156), (0.781, 0.125),   # céu azul
                  (0.234, 0.583), (0.117, 0.625), (0.570, 0.635),   # encosta e telhados
                  (0.469, 0.833), (0.883, 0.271)],                  # asfalto e a placa que oclui
    "rois": {"ceu": (0.05, 0.05, 0.30, 0.20), "nuvem_haze": (0.60, 0.50, 0.78, 0.56),
             "edificacao": (0.02, 0.56, 0.22, 0.66)},
    # referência traçada à mão sobre a imagem (aproximada — ver aviso na Etapa 4)
    "referencia": [(0.305, 0.552), (0.316, 0.490), (0.328, 0.448), (0.352, 0.425), (0.391, 0.420),
                   (0.426, 0.442), (0.438, 0.408), (0.469, 0.375), (0.516, 0.359), (0.563, 0.359),
                   (0.609, 0.344), (0.648, 0.313), (0.688, 0.297), (0.727, 0.286), (0.773, 0.292),
                   (0.820, 0.302), (0.875, 0.302), (0.938, 0.297), (0.996, 0.302), (0.998, 0.392),
                   (0.930, 0.460), (0.855, 0.500), (0.781, 0.530), (0.703, 0.545), (0.625, 0.540),
                   (0.547, 0.526), (0.484, 0.516), (0.430, 0.521), (0.359, 0.536)],
}

if CFG["IMAGEM"] == "socal":
    PRESET, CAMINHO = dict(CALIBRADO), "pluma_socal.jpg"
    if not os.path.exists(CAMINHO):
        req = urllib.request.Request(PRESET["url"], headers={"User-Agent": "colab-sam/1.0"})
        open(CAMINHO, "wb").write(urllib.request.urlopen(req).read())
elif CFG["IMAGEM"] == "upload":
    from google.colab import files
    PRESET, CAMINHO = {}, list(files.upload().keys())[0]
elif str(CFG["IMAGEM"]).startswith("http"):
    PRESET, CAMINHO = {}, "imagem_usuario.jpg"
    req = urllib.request.Request(CFG["IMAGEM"], headers={"User-Agent": "colab-sam/1.0"})
    open(CAMINHO, "wb").write(urllib.request.urlopen(req).read())
else:
    PRESET, CAMINHO = {}, CFG["IMAGEM"]

IMG = cv2.cvtColor(cv2.imread(CAMINHO), cv2.COLOR_BGR2RGB)   # cv2 lê pelo conteúdo, não pela extensão
ALT, LARG = IMG.shape[:2]
print(f"imagem {LARG}x{ALT} | dispositivo={DISPOSITIVO} | modelo={MODELO}")


def para_px(pts):
    p = np.asarray(pts, float).reshape(-1, 2)
    return np.stack([p[:, 0] * LARG, p[:, 1] * ALT], axis=1)


def caixa_px(c):
    return np.array([c[0] * LARG, c[1] * ALT, c[2] * LARG, c[3] * ALT], float)


def roi_mascara(c):
    x0, y0, x1, y1 = caixa_px(c).astype(int)
    m = np.zeros((ALT, LARG), bool); m[max(y0, 0):y1, max(x0, 0):x1] = True
    return m


def poligono_mascara(v):
    m = np.zeros((ALT, LARG), np.uint8)
    cv2.fillPoly(m, [para_px(v).astype(np.int32)], 1)
    return m.astype(bool)


def palpite():
    """Palpite de caixa e pontos positivos para uma imagem sem prompts calibrados.

    Fumaça iluminada é CLARA e pouco SATURADA, e a pluma fica acima da linha do solo.
    Não é um detector — é só um ponto de partida, e já mostra o problema do enunciado:
    o mesmo critério também pega céu claro e nuvem.
    """
    hsv = cv2.cvtColor(IMG, cv2.COLOR_RGB2HSV)
    cand = (hsv[:, :, 2] > 110) & (hsv[:, :, 1] < 80)
    cand[int(0.75 * ALT):] = False
    cand = cv2.morphologyEx(cand.astype(np.uint8), cv2.MORPH_OPEN, np.ones((9, 9), np.uint8))
    n, rot, stats, _ = cv2.connectedComponentsWithStats(cand, 8)
    if n < 2:
        return (0.25, 0.20, 0.95, 0.60), [(0.45, 0.35), (0.55, 0.40), (0.65, 0.35)]
    i = 1 + int(np.argmax(stats[1:, cv2.CC_STAT_AREA]))
    x, y, w, h = stats[i, :4]
    ys, xs = np.nonzero(rot == i)
    pos = [(float(np.quantile(xs, q) / LARG),
            float(np.median(ys[np.abs(xs - np.quantile(xs, q)) < max(0.02 * LARG, 2)]) / ALT))
           for q in (0.25, 0.5, 0.75)]
    return (x / LARG, y / ALT, (x + w) / LARG, (y + h) / ALT), pos


if PRESET:
    PROMPTS = {k: PRESET[k] for k in ("caixa", "positivos", "negativos")}
    ROIS = {n: roi_mascara(c) for n, c in PRESET["rois"].items()}
    REFERENCIA = poligono_mascara(PRESET["referencia"])
else:
    caixa, pos = palpite()
    PROMPTS = {"caixa": caixa, "positivos": pos,
               "negativos": [(0.04, 0.04), (0.96, 0.04), (0.50, 0.96), (0.04, 0.90)]}
    ROIS = {"ceu": roi_mascara((0.0, 0.0, 1.0, 0.12)), "solo": roi_mascara((0.0, 0.88, 1.0, 1.0))}
    REFERENCIA = None
    print("ATENÇÃO: imagem sem calibração. Palpite:", {k: np.round(v, 3).tolist()
                                                       for k, v in PROMPTS.items()})

# imagem com grade de coordenadas, para conferir e ajustar os prompts à mão
fig, ax = plt.subplots(figsize=(11, 11 * ALT / LARG))
ax.imshow(IMG)
for x in range(0, LARG, 100):
    ax.axvline(x, color="yellow", lw=.4); ax.text(x + 3, 16, f"{x}\n{x/LARG:.2f}", color="red", fontsize=6, va="top")
for y in range(0, ALT, 100):
    ax.axhline(y, color="yellow", lw=.4); ax.text(3, y + 6, f"{y} / {y/ALT:.2f}", color="red", fontsize=6, va="top")
if REFERENCIA is not None:
    ax.contour(REFERENCIA, levels=[0.5], colors="deepskyblue", linewidths=2)
ax.set_title(f"{CFG['IMAGEM']} — {LARG}x{ALT}px (rótulos: pixel / fração; azul = referência)")
ax.axis("off"); fig.tight_layout()
fig.savefig("saida/01_grade.png", bbox_inches="tight"); plt.show()

---
# Etapa 2 — Três configurações de *prompt*

| | *Prompt* | Hipótese testada |
|---|---|---|
| **A** | só a caixa | o retângulo basta? Ele contém muito céu junto da pluma. |
| **B** | só pontos positivos | sem limite espacial, até onde a região "espalha" por semelhança? |
| **C** | caixa + positivos + **negativos** | dizer o que **não** é pluma resolve a ambiguidade? |

**Justificativa da melhoria.** O SAM é agnóstico a classe: ele não tem o conceito "fumaça", só
devolve a região coerente que o *prompt* indica. Por isso o caminho do notebook inicial — deixar um
detector achar a fumaça — falhou duas vezes: o detector não detectou nada e o *prompt* de texto
exigia um modelo indisponível. Com *prompt* geométrico o controle volta para as nossas mãos, e o
**ponto negativo** passa a ser o único elemento capaz de excluir um distrator que está *dentro* da
caixa — exatamente o caso do céu claro, do *haze* e do telhado.

Melhoria de código: `set_image()` é chamado **uma vez** e reaproveitado pelas três configurações. O
custo do SAM está em codificar a imagem; o decodificador de *prompt* é leve.

In [ ]:
# ============================================================
# ETAPA 2 — Predição
# ============================================================
from ultralytics.models.sam import SAM2Predictor

predictor = SAM2Predictor(overrides=dict(task="segment", mode="predict", model=MODELO,
                                         save=False, verbose=False, conf=0.25))
predictor.set_image(CAMINHO)


def prever(caixa=None, pontos=None, rotulos=None):
    """Chama o SAM com prompt geométrico e devolve uma máscara booleana HxW."""
    kw = {}
    if caixa is not None:
        kw["bboxes"] = caixa_px(caixa)[None, :]
    if pontos:
        kw["points"] = para_px(pontos)
        kw["labels"] = np.asarray(rotulos, int)
    r = predictor(**kw)[0]
    if r.masks is None or len(r.masks.data) == 0:
        return np.zeros((ALT, LARG), bool)
    m = r.masks.data.cpu().numpy().any(0).astype(np.uint8)
    if m.shape != (ALT, LARG):               # defesa: o notebook inicial também reescalava
        m = cv2.resize(m, (LARG, ALT), interpolation=cv2.INTER_NEAREST)
    return m.astype(bool)


n_pos, n_neg = len(PROMPTS["positivos"]), len(PROMPTS["negativos"])
CONFIGS = {
    "A — só caixa": dict(caixa=PROMPTS["caixa"]),
    "B — só pontos (+)": dict(pontos=PROMPTS["positivos"], rotulos=np.ones(n_pos, int)),
    "C — caixa + (+) + (−)": dict(caixa=PROMPTS["caixa"],
                                  pontos=list(PROMPTS["positivos"]) + list(PROMPTS["negativos"]),
                                  rotulos=np.r_[np.ones(n_pos, int), np.zeros(n_neg, int)]),
}

MASCARAS = {nome: prever(**cfg) for nome, cfg in CONFIGS.items()}
for nome, m in MASCARAS.items():
    print(f"{nome:24s} área = {100*m.mean():5.2f}% da imagem")

---
# Etapa 3 — Apresentação da máscara

Original com os *prompts* desenhados, máscara binária e máscara sobreposta, lado a lado, para cada
configuração. Três decisões de apresentação:

* os *prompts* vão **sobre a original** — sem isso não dá para saber se o erro é do modelo ou de um
  ponto mal colocado;
* a sobreposição usa **transparência + contorno**, porque preenchimento opaco esconde exatamente a
  borda, que é onde está o erro interessante;
* o painel binário traz o **percentual de área**, para dar escala ao que se vê.

In [ ]:
# ============================================================
# ETAPA 3 — Figuras lado a lado
# ============================================================

def sobrepor(img, mask, alpha=0.45):
    out = img.astype(np.float32).copy()
    out[mask] = (1 - alpha) * out[mask] + alpha * np.array([255, 60, 60], np.float32)
    out = out.astype(np.uint8)
    cont, _ = cv2.findContours(mask.astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
    cv2.drawContours(out, cont, -1, (255, 230, 0), 2)
    return out


fig, axs = plt.subplots(3, 3, figsize=(13, 3.2 * 3 * ALT / LARG + 1.2))
for lin, (nome, cfg) in enumerate(CONFIGS.items()):
    m = MASCARAS[nome]
    axs[lin, 0].imshow(IMG)
    if cfg.get("caixa") is not None:
        bx0, by0, bx1, by1 = caixa_px(cfg["caixa"])
        axs[lin, 0].add_patch(Rectangle((bx0, by0), bx1 - bx0, by1 - by0, fill=False,
                                        ec="yellow", lw=2))
    if cfg.get("pontos"):
        pp, rr = para_px(cfg["pontos"]), np.asarray(cfg["rotulos"])
        axs[lin, 0].scatter(pp[rr == 1, 0], pp[rr == 1, 1], c="lime", marker="*", s=150, ec="k", lw=.6)
        axs[lin, 0].scatter(pp[rr == 0, 0], pp[rr == 0, 1], c="red", marker="X", s=90, ec="k", lw=.6)
    axs[lin, 1].imshow(m, cmap="gray")
    axs[lin, 2].imshow(sobrepor(IMG, m))
    axs[lin, 0].set_ylabel(nome, fontsize=9)
    axs[lin, 1].set_xlabel(f"área = {100*m.mean():.2f}%", fontsize=8)
    for a in axs[lin]:
        a.set_xticks([]); a.set_yticks([])
for c, t in enumerate(["original + prompts (★ positivo, ✕ negativo)", "máscara binária", "sobreposição"]):
    axs[0, c].set_title(t)
fig.tight_layout(); fig.savefig("saida/03_comparativo.png", bbox_inches="tight"); plt.show()

---
# Etapa 4 — Avaliação no código

| Pergunta do enunciado | Medida |
|---|---|
| A máscara **cobre** a pluma? | `recall` contra a referência |
| **Inclui** o que não pertence a ela? | `precisão` e `invadiu ceu / nuvem_haze / edificacao (%)` |
| **Deixa fumaça de fora?** | `1 − recall` e o mapa de erro (azul) |
| O resultado é **estável**? | `IoU` entre as três configurações: quanto menor, mais a resposta veio do *prompt* e não da imagem |
| A borda corresponde a algo real? | **separabilidade dentro × fora** (Bhattacharyya): se for baixa, a máscara cortou uma região fotometricamente homogênea |

> ⚠️ A máscara de referência é um polígono traçado **à mão** e **aproximado** — na base da pluma não
> existe fronteira objetiva entre fumaça e *haze*. Os valores de IoU/Dice servem para comparar as
> configurações **entre si**, não como medida absoluta. Essa incerteza do próprio rótulo é parte da
> resposta à pergunta "por que é difícil".

A Etapa 4.2 fecha a pergunta sobre **iluminação e contraste**: a mesma cena, o mesmo *prompt*, com o
contraste reduzido à metade — como o SAM recalcula o *embedding*, a máscara muda sem que nada na cena
tenha mudado.

In [ ]:
# ============================================================
# ETAPA 4.1 — Métricas
# ============================================================
CINZA = cv2.cvtColor(IMG, cv2.COLOR_RGB2GRAY)
SAT = cv2.cvtColor(IMG, cv2.COLOR_RGB2HSV)[:, :, 1]


def bhattacharyya(a, b, bins=64):
    """0 = distribuições idênticas; 8 = saturado (histogramas sem sobreposição)."""
    if len(a) == 0 or len(b) == 0:
        return float("nan")
    faixa = (float(min(a.min(), b.min())), float(max(a.max(), b.max())) + 1e-6)
    ha = np.histogram(a, bins, faixa)[0]; hb = np.histogram(b, bins, faixa)[0]
    ha = ha / (ha.sum() + 1e-12); hb = hb / (hb.sum() + 1e-12)
    return float(min(-np.log(np.sum(np.sqrt(ha * hb)) + 1e-12), 8.0))


def iou(a, b):
    u = np.logical_or(a, b).sum()
    return float(np.logical_and(a, b).sum() / u) if u else 0.0


def metricas(m):
    d = {"área (%)": 100 * m.mean()}
    n, _, stats, _ = cv2.connectedComponentsWithStats(m.astype(np.uint8), 8)
    d["componentes"] = int(n - 1)
    cont, _ = cv2.findContours(m.astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
    per = sum(cv2.arcLength(c, True) for c in cont)
    d["compacidade"] = float(4 * np.pi * m.sum() / per ** 2) if per else 0.0
    for nome, roi in ROIS.items():
        d[f"invadiu {nome} (%)"] = 100 * np.logical_and(m, roi).sum() / roi.sum()
    d["separab. luminância"] = bhattacharyya(CINZA[m].astype(float), CINZA[~m].astype(float))
    d["separab. saturação"] = bhattacharyya(SAT[m].astype(float), SAT[~m].astype(float))
    if REFERENCIA is not None:
        vp = np.logical_and(m, REFERENCIA).sum()
        d["IoU"] = iou(m, REFERENCIA)
        d["Dice"] = float(2 * vp / (m.sum() + REFERENCIA.sum())) if m.sum() else 0.0
        d["precisão"] = float(vp / m.sum()) if m.sum() else 0.0
        d["recall"] = float(vp / REFERENCIA.sum())
    return d


TABELA = pd.DataFrame({n: metricas(m) for n, m in MASCARAS.items()}).T.round(3)
display(TABELA)
TABELA.to_csv("saida/04_metricas.csv")

nomes = list(MASCARAS)
for i, a in enumerate(nomes):
    for b in nomes[i + 1:]:
        print(f"IoU({a.split(' ')[0]}, {b.split(' ')[0]}) = {iou(MASCARAS[a], MASCARAS[b]):.3f}")

if REFERENCIA is not None:
    melhor = TABELA["IoU"].idxmax()
    mm = MASCARAS[melhor]
    mapa = np.zeros((ALT, LARG, 3), np.uint8)
    mapa[np.logical_and(mm, REFERENCIA)] = (60, 200, 60)
    mapa[np.logical_and(mm, ~REFERENCIA)] = (230, 70, 70)
    mapa[np.logical_and(~mm, REFERENCIA)] = (70, 120, 240)
    fig, ax = plt.subplots(figsize=(9, 9 * ALT / LARG))
    ax.imshow((IMG * 0.45 + mapa * 0.55).astype(np.uint8)); ax.axis("off")
    ax.set_title(f"Mapa de erro — {melhor}\nverde = acerto | vermelho = incluiu o que não é pluma"
                 " | azul = deixou fumaça de fora")
    fig.tight_layout(); fig.savefig("saida/04_erro.png", bbox_inches="tight"); plt.show()

In [ ]:
# ============================================================
# ETAPA 4.2 — Efeito do contraste (mesma cena, mesmo prompt)
# ============================================================
base = MASCARAS["C — caixa + (+) + (−)"]
x = IMG.astype(np.float32) / 255
baixo = (np.clip((x - 0.5) * 0.5 + 0.5, 0, 1) * 255).astype(np.uint8)
cv2.imwrite("saida/_baixo_contraste.png", cv2.cvtColor(baixo, cv2.COLOR_RGB2BGR))

predictor.set_image("saida/_baixo_contraste.png")     # reencoda a imagem alterada
m2 = prever(**CONFIGS["C — caixa + (+) + (−)"])
predictor.set_image(CAMINHO)                          # restaura
print(f"contraste x0.5 -> IoU vs original = {iou(m2, base):.3f} | "
      f"área {100*(m2.mean()-base.mean()):+.2f} p.p.")

fig, axs = plt.subplots(1, 2, figsize=(13, 13 * ALT / LARG / 2))
axs[0].imshow(sobrepor(IMG, base)); axs[0].set_title("original (C)")
axs[1].imshow(sobrepor(baixo, m2))
axs[1].set_title(f"contraste reduzido à metade\nIoU vs original = {iou(m2, base):.2f}")
for a in axs:
    a.axis("off")
fig.tight_layout(); fig.savefig("saida/04_contraste.png", bbox_inches="tight"); plt.show()

for nome, m in MASCARAS.items():
    rot = nome.split(" ")[0]
    cv2.imwrite(f"saida/mascara_{rot}.png", (m * 255).astype(np.uint8))
    cv2.imwrite(f"saida/sobreposicao_{rot}.png", cv2.cvtColor(sobrepor(IMG, m), cv2.COLOR_RGB2BGR))
print("gerado:", sorted(os.listdir("saida")))

---
# Etapa 5 — Conclusões

*(Confira os números da **sua** execução nas tabelas acima antes de entregar.)*

## A máscara cobre a pluma? Inclui o que não é? Deixa fumaça de fora?

* **A (só caixa)** — o retângulo que contém a pluma contém também muito céu e parte do horizonte. O
  SAM devolve a região mais coerente dentro dele, então a máscara ou vaza para o céu (veja
  `invadiu ceu (%)` e a `precisão`) ou encolhe para o núcleo mais denso, perdendo as bordas finas
  (queda no `recall`).
* **B (só pontos positivos)** — sem limite espacial, a máscara cresce por continuidade de aparência e
  tende a engolir a faixa de *haze* do horizonte, que tem praticamente a mesma estatística da base da
  pluma. É o que as colunas `área (%)` e `invadiu nuvem_haze (%)` mostram.
* **C (caixa + positivos + negativos)** — os negativos em céu, *haze*, telhado, encosta e placa são o
  único elemento do *prompt* capaz de excluir um distrator que está **dentro** da caixa. É a
  configuração que se espera melhor; as colunas de invasão e a `precisão` dizem se foi o que
  aconteceu.

O **IoU entre as três** é a medida mais honesta do problema: se ele for baixo, três *prompts*
razoáveis sobre a mesma imagem produziram três respostas diferentes — ou seja, a resposta veio do
*prompt*, não da imagem.

## O que dificultou a segmentação

* **Semelhança fumaça × nuvem/haze.** Fator dominante, e aparece na `separab. saturação`: fumaça e
  nuvem são ambas quase acromáticas, então cor não separa as duas. O palpite automático da Etapa 1
  (claro + pouco saturado) pega céu e nuvem junto — é a demonstração prática disso.
* **Contraste e borda difusa.** Fumaça é um meio **semitransparente**: perto da borda cada pixel é
  mistura de fumaça e fundo. Não há gradiente forte para o contorno se apoiar, e por isso a borda
  acaba decidida pela caixa que desenhamos. A `separab. luminância` quantifica isso: valor baixo
  significa que a máscara cortou uma região fotometricamente homogênea.
* **Iluminação.** A Etapa 4.2 mostra a mesma cena, com o mesmo *prompt*, dando outra máscara só
  porque o contraste mudou — o SAM recalcula o *embedding* e passa a "ver" outra coisa. Consequência
  prática: fotos do mesmo incêndio em horários diferentes não são comparáveis sem normalização.
* **Oclusão e contexto urbano.** A placa de trânsito corta a pluma em duas; sem um ponto negativo
  sobre ela, ela é absorvida pela máscara. Telhados e encosta competem na parte inferior da caixa.
* **O rótulo também é incerto.** A referência traçada à mão discorda de si mesma na base da pluma.
  Nenhum IoU aqui deve ser lido como "erro do modelo" sem essa ressalva.
* **Detecção automática não resolve.** O notebook inicial tentou dois atalhos automáticos e os dois
  falharam. Segmentar fumaça com SAM é, hoje, uma tarefa guiada: alguém precisa dizer onde ela está.

## O que eu faria a seguir

* **Refinamento iterativo** realimentando a máscara anterior como `masks=` no *prompt*, e limpeza
  morfológica para remover ilhas isoladas — com o cuidado de que manter só o maior componente conexo
  descartaria fumaça real onde a placa oclui a pluma.
* Sair da máscara binária para **estimativa de transparência (alfa)**, formato adequado a um meio
  semitransparente e a medir área de pluma em série temporal.
* **Referência de consenso** (duas ou três pessoas traçando a mesma pluma): a concordância entre elas
  é o teto realista de IoU que qualquer modelo pode atingir nesta imagem.
* Repetir o protocolo em **N imagens** e reportar média ± desvio, em vez de um caso só.

---

## Entrega

Rode tudo, depois `Arquivo → Salvar uma cópia no GitHub`. O link a entregar é o do `.ipynb` no
GitHub. Rodar **antes** de salvar importa: o GitHub renderiza o notebook com as saídas embutidas,
então o professor vê as figuras e as tabelas sem precisar executar nada.